# Telecom Churn Analysis — Legacy Notebook

This notebook is a **self-contained, old-school data science workflow**.
Everything is done inline — no `src/` imports. This is how we used to do it
before modularizing into a production pipeline.

**Sections:** Data Loading → EDA → Cleaning → Feature Engineering → Train/Test Split → Model Training → Evaluation → Inference

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = REPO_ROOT / "data" / "raw" / "telecom_churn.csv"
print(f"Data path: {DATA_PATH}")

---
## 1. Data Loading

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

In [ ]:
# Check for missing values
missing = df_raw.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values.")

In [ ]:
# Check for duplicates
n_dupes = df_raw.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")

In [ ]:
# Target distribution
churn_counts = df_raw["Churn"].value_counts()
churn_pct = df_raw["Churn"].value_counts(normalize=True) * 100

print("Churn distribution:")
print(churn_counts)
print(f"\nChurn rate: {churn_pct[1]:.1f}%")

fig, ax = plt.subplots(figsize=(5, 4))
churn_counts.plot(kind="bar", color=["steelblue", "salmon"], ax=ax)
ax.set_title("Churn Distribution")
ax.set_xlabel("Churn (0=No, 1=Yes)")
ax.set_ylabel("Count")
ax.set_xticklabels(["No", "Yes"], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Distributions of numeric features
numeric_cols = ["AccountWeeks", "DataUsage", "CustServCalls", "DayMins",
                "DayCalls", "MonthlyCharge", "OverageFee", "RoamMins"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, col in enumerate(numeric_cols):
    ax = axes[i // 4, i % 4]
    df_raw[col].hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(col)
plt.suptitle("Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = df_raw.corr(numeric_only=True)
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
fig.colorbar(im, ax=ax)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by key categorical features
for col in ["ContractRenewal", "DataPlan"]:
    ct = df_raw.groupby(col)["Churn"].mean() * 100
    print(f"\nChurn rate by {col}:")
    print(ct.round(1))

In [ ]:
# Boxplots: numeric features by churn
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, col in enumerate(numeric_cols):
    ax = axes[i // 4, i % 4]
    df_raw.boxplot(column=col, by="Churn", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("Churn")
plt.suptitle("Features by Churn", fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Data Cleaning

In [ ]:
df = df_raw.copy()

# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)

# Drop duplicates
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows.")

# Replace sentinel missing values
df = df.replace(["NA", "N/A", "", "?", "null", "None", "missing", -999], pd.NA)

# Drop any remaining NAs
before = len(df)
df = df.dropna()
print(f"Dropped {before - len(df)} rows with missing values.")

# Reset index
df = df.reset_index(drop=True)

print(f"\nClean dataset shape: {df.shape}")
df.head()

---
## 4. Feature Engineering

In [ ]:
TARGET = "churn"

# Separate target
y = df[TARGET].copy()
X = df.drop(columns=[TARGET]).copy()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")

---
## 5. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} rows")
print(f"Test set:  {X_test.shape[0]} rows")
print(f"\nTrain churn rate: {y_train.mean():.3f}")
print(f"Test churn rate:  {y_test.mean():.3f}")

---
## 6. Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LogisticRegression(solver="liblinear", max_iter=500, random_state=42)

# Quick cross-validation on training set
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
print(f"5-fold CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Fit on full training set
model.fit(X_train, y_train)
print("Model fitted.")

In [ ]:
# Save model artifact
import joblib

model_dir = REPO_ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "model_legacy.pkl"
joblib.dump(model, model_path)
print(f"Model saved to: {model_path}")

---
## 7. Evaluation

In [ ]:
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)

y_pred = model.predict(X_test)

f1 = f1_score(y_test, y_pred, average="weighted")
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy:          {acc:.4f}")
print(f"Weighted F1 Score: {f1:.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
fig.colorbar(im, ax=ax)

# Annotate cells
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")

ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["No Churn", "Churn"])
ax.set_yticklabels(["No Churn", "Churn"])
plt.tight_layout()
plt.show()

---
## 8. Inference

In [ ]:
# Run predictions on test set (simulating new data)
predictions = pd.DataFrame({
    "prediction": model.predict(X_test),
}, index=X_test.index)

# Add probabilities if available
if hasattr(model, "predict_proba"):
    predictions["churn_probability"] = model.predict_proba(X_test)[:, 1]

# Map labels
predictions["label"] = predictions["prediction"].map({0: "No Churn", 1: "Churn"})

print(f"Predictions shape: {predictions.shape}")
predictions.head(10)

In [ ]:
# Save predictions
output_dir = REPO_ROOT / "data" / "inference"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "predictions_legacy.csv"
predictions.to_csv(output_path, index=False)
print(f"Predictions saved to: {output_path}")

---
## Summary

This legacy notebook ran the full churn-prediction workflow in a single file:

1. **Loaded** the raw CSV manually
2. **Explored** the data with descriptive stats and plots
3. **Cleaned** column names, duplicates, and missing values inline
4. **Engineered** features (minimal — used raw numeric + categorical columns)
5. **Split** into train/test with stratification
6. **Trained** a Logistic Regression model
7. **Evaluated** with F1, accuracy, classification report, and confusion matrix
8. **Predicted** on the test set with probabilities

This approach works for quick experiments but becomes hard to maintain,
test, and deploy at scale. The production `src/` modules solve this.